In [1]:
!git clone https://github.com/dave123981/commerce-recommendation-engine.git
%cd commerce-recommendation-engine
!pip install -r requirements.txt -q

Cloning into 'commerce-recommendation-engine'...
remote: Enumerating objects: 145, done.
remote: Counting objects: 100% (145/145), done.
remote: Compressing objects: 100% (106/106), done.
remote: Total 145 (delta 72), reused 78 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (145/145), 49.38 KiB | 1.70 MiB/s, done.
Resolving deltas: 100% (72/72), done.
/content/commerce-recommendation-engine


In [2]:
!pwd

!ls

/content/commerce-recommendation-engine
data  notebooks  README.md  requirements.txt  src  tests


In [3]:
!pip install -r requirements.txt

In [4]:
!pip install -q kagglehub
import kagglehub

!python data/download_data.py          # kagglehub is now the default method
!python data/build_interactions.py

100% 197M/197M [00:01<00:00, 152MB/s]
Extracting files...
kagglehub cached the dataset at /root/.cache/kagglehub/datasets/psparks/instacart-market-basket-analysis/versions/1
Copied CSVs to /content/commerce-recommendation-engine/data/raw
Built 31,803,792 interactions | 175,071 users | 20,094 items | 3,223,536 orders
Saved to /content/commerce-recommendation-engine/data/interactions.csv


In [5]:
import pandas as pd

interactions = pd.read_csv("data/interactions.csv", parse_dates=["timestamp"])

print(interactions.shape)
print(interactions.nunique())
print("sparsity:", 1 - len(interactions) / (interactions.user_id.nunique() * interactions.item_id.nunique()))
print("orders per user:", interactions.groupby("user_id").order_id.nunique().describe())
print("purchases per item:", interactions.groupby("item_id").order_id.nunique().describe())

(31803792, 5)
user_id       175071
item_id        20094
order_id     3223536
timestamp        366
quantity           1
dtype: int64
sparsity: 0.9909593783559679
orders per user: count    175071.000000
mean         18.412735
std          17.131480
min           1.000000
25%           7.000000
50%          12.000000
75%          23.000000
max         100.000000
Name: order_id, dtype: float64
purchases per item: count     20094.000000
mean       1582.750672
std        7520.228193
min         100.000000
25%         183.000000
50%         371.000000
75%        1014.000000
max      476386.000000
Name: order_id, dtype: float64


In [6]:
import data.build_interactions as bi
interactions = bi.build_interactions(min_orders_per_user=10, min_purchases_per_item=100)
interactions.to_csv("data/interactions.csv", index=False)

In [7]:
from src.metrics import time_based_split

train, test = time_based_split(interactions, test_frac=0.2)
train.to_csv("data/train.csv", index=False)
test.to_csv("data/test.csv", index=False)
print(len(train), len(test))

21931935 5429120


In [10]:
from src.v2_association import AssociationRecommender
from src.metrics import evaluate_model

model_v2 = AssociationRecommender(min_support=0.0005, min_confidence=0.2, max_items=2000).fit(train)
metrics = evaluate_model(model_v2, test, k=10)
print(metrics)

{'precision@10': 0.04267989612517103, 'recall@10': 0.015335332511294807, 'map@10': 0.016411354679002784, 'ndcg@10': 0.04574392270160971}


In [11]:
n_items_with_rules = len(model_v2._rules_map)
n_items_total = train.item_id.nunique()
print(f"items with at least one rule: {n_items_with_rules} / {n_items_total} ({n_items_with_rules/n_items_total:.1%})")

users_with_a_rule = sum(
    1 for u in test.user_id.unique()
    if any(i in model_v2._rules_map for i in model_v2._user_items.get(u, set()))
)
print(f"test users with at least one antecedent rule available: {users_with_a_rule} / {test.user_id.nunique()} ({users_with_a_rule/test.user_id.nunique():.1%})")

items with at least one rule: 35 / 18449 (0.2%)
test users with at least one antecedent rule available: 55621 / 107437 (51.8%)


### Optimize `min_support` and `min_confidence`

Let's find the optimal `min_support` and `min_confidence` values by iterating through different combinations and evaluating the model's performance for each.

In [ ]:
import numpy as np

min_supports = np.linspace(0.0001, 0.001, 5) # Example range for min_support
min_confidences = np.linspace(0.1, 0.4, 4) # Example range for min_confidence

results = []

for ms in min_supports:
    for mc in min_confidences:
        print(f"Testing with min_support={ms:.5f}, min_confidence={mc:.2f}")
        try:
            model = AssociationRecommender(min_support=ms, min_confidence=mc, max_items=2000).fit(train)
            metrics = evaluate_model(model, test, k=10)
            results.append({
                'min_support': ms,
                'min_confidence': mc,
                'precision@10': metrics.get('precision@10'),
                'recall@10': metrics.get('recall@10'),
                'map@10': metrics.get('map@10'),
                'ndcg@10': metrics.get('ndcg@10')
            })
        except ValueError as e:
            print(f"  No rules found for min_support={ms:.5f}, min_confidence={mc:.2f}: {e}")
            results.append({
                'min_support': ms,
                'min_confidence': mc,
                'precision@10': 0.0,
                'recall@10': 0.0,
                'map@10': 0.0,
                'ndcg@10': 0.0 # Assign 0 or NaN if no rules are found
            })

results_df = pd.DataFrame(results)
print("\nOptimization Results:")
display(results_df.sort_values(by='map@10', ascending=False))


Testing with min_support=0.00010, min_confidence=0.10
